In [88]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [89]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [90]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import HTML
from thinkstats import decorate

**Reading the Data**

In [91]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

In [92]:
try:
    import statadict
except ImportError:
    %pip install statadict

In [93]:
dct_file = "2002FemPreg.dct"
dat_file = "2002FemPreg.dat.gz"

In [94]:
from statadict import parse_stata_dict

def read_stata(dct_file, dat_file):
  stata_dict = parse_stata_dict(dct_file)
  resp = pd.read_fwf(
      dat_file,
      names=stata_dict.names,
      colspecs=stata_dict.colspecs,
      compression="gzip"
  )
  return resp

In [95]:
preg = read_stata(dct_file, dat_file)

In [96]:
preg.shape

(13593, 243)

In [97]:
preg.head()

,caseid,pregordr,howpreg_n,howpreg_p,moscurrp,nowprgdk,pregend1,pregend2,nbrnaliv,multbrth,...,poverty_i,laborfor_i,religion_i,metro_i,basewgt,adj_mod_basewgt,finalwgt,secu_p,sest,cmintvw
0,1,1,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,3410.389399,3869.349602,6448.271112,2,9,1231
1,1,2,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,3410.389399,3869.349602,6448.271112,2,9,1231
2,2,1,NaN,NaN,NaN,NaN,5.0,NaN,3.0,5.0,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231
3,2,2,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231
4,2,3,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231


In [98]:
preg.columns

Index(['caseid', 'pregordr', 'howpreg_n', 'howpreg_p', 'moscurrp', 'nowprgdk',
       'pregend1', 'pregend2', 'nbrnaliv', 'multbrth',
       ...
       'poverty_i', 'laborfor_i', 'religion_i', 'metro_i', 'basewgt',
       'adj_mod_basewgt', 'finalwgt', 'secu_p', 'sest', 'cmintvw'],
      dtype='object', length=243)

In [99]:
pregordr = preg["pregordr"]
type(pregordr)

pandas.core.series.Series

In [100]:
pregordr.head()

,pregordr
0,1
1,2
2,1
3,2
4,3


**Validation**

In [101]:
preg["outcome"].value_counts().sort_index()

,count
outcome,
1,9148
2,1862
3,120
4,1921
5,190
6,352


In [102]:
counts = preg["birthwgt_lb"].value_counts(dropna=False).sort_index()
counts

,count
birthwgt_lb,
0.0,8
1.0,40
2.0,53
3.0,98
4.0,229
5.0,697
6.0,2223
7.0,3049
8.0,1889


In [103]:
counts.loc[0:5].sum()

np.int64(1125)

In [104]:
preg["birthwgt_lb"] = preg["birthwgt_lb"].replace([51, 97, 98, 99], np.nan)

**Transformation**

In [105]:
preg["agepreg"] /= 100.0
preg["agepreg"].mean()

np.float64(24.6881511970395)

In [106]:
preg["birthwgt_oz"].value_counts(dropna=False).sort_index()

,count
birthwgt_oz,
0.0,1037
1.0,408
2.0,603
3.0,533
4.0,525
5.0,535
6.0,709
7.0,501
8.0,756


In [107]:
preg["birthwgt_oz"] = preg["birthwgt_lb"].replace([97, 98, 99], np.nan)

In [108]:
preg["totalwgt_lb"] = preg["birthwgt_lb"] + preg["birthwgt_oz"] / 16.0
preg["totalwgt_lb"].mean()

np.float64(7.2591300638485245)

**Summary Statistics**

In [109]:
weights = preg["totalwgt_lb"]
n = weights.count()
n

np.int64(9084)

In [110]:
mean = weights.sum() / n
mean

np.float64(7.2591300638485245)

In [111]:
squared_deviations = (weights - mean) ** 2

In [112]:
var = squared_deviations.sum() / n
var

np.float64(2.248738832736632)

In [113]:
weights.var()

2.248986409399892

In [114]:
weights.var(ddof=0)

2.2487388327365943

In [115]:
std = np.sqrt(var)
std

np.float64(1.4995795519867001)

In [116]:
weights.std(ddof=0)

1.4995795519866875

**Interpretation**


In [117]:
subset = preg.query("caseid == 10229")
subset.shape

(7, 244)

In [118]:
subset["outcome"].values

array([4, 4, 4, 4, 4, 4, 1])

**To work with data effectively, you have to think on two levels at the same time: The level of statistics and the level of context.**

The outcome code 1 indicates a live birth. Code 4 indicates a miscarriage – that is, a pregnancy loss, usually with no known medical cause.

Statistically this respondent is not unusual. Pregnancy loss is common and there are other respondents who reported as many instances. But remembering the context, this data tells the story of a woman who was pregnant six times, each time ending in miscarriage. Her seventh and most recent pregnancy ended in a live birth. If we consider this data with empathy, it is natural to be moved by the story it tells.

Each row in the NSFG dataset represents a person who provided honest answers to many personal and difficult questions. We can use this data to answer statistical questions about family life, reproduction, and health. At the same time, we have an obligation to consider the people represented by the data, and to afford them respect and gratitude.

**Exercise 1**

In [119]:
preg["birthord"].value_counts(dropna=False).sort_index()

,count
birthord,
1.0,4413
2.0,2874
3.0,1234
4.0,421
5.0,126
6.0,50
7.0,20
8.0,7
9.0,2


**Exercise 2**

In [120]:
preg["totalwgt_kg"] = preg["totalwgt_lb"] / 2.2
preg["totalwgt_kg"]

,totalwgt_kg
0,3.863636
1,3.380682
2,4.346591
3,3.380682
4,2.897727
...,...
13588,2.897727
13589,NaN
13590,NaN
13591,3.380682


In [121]:
preg["totalwgt_kg"].mean()

np.float64(3.299604574476602)

In [122]:
preg["totalwgt_kg"].std(ddof=0)

0.6816270690848191

**Exercise 3**

In [123]:
preg.query("caseid == 2298")["prglngth"]

,prglngth
2610,40
2611,36
2612,30
2613,40


In [124]:
preg.query("caseid == 5013 and birthord == 1.0")["birthwgt_lb"]

,birthwgt_lb
5516,7.0
